In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
import random
import string
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

print('Library berhasil diimport!')

In [ ]:
FILE_PATH = 'DataIuranWarga07.xlsx'
IURAN_BULANAN = 50000
PERIODE_AKHIR = pd.Timestamp('2025-12-31')

df = pd.read_excel(FILE_PATH)
df.columns = ['Nama', 'Tanggal Bayar', 'Nominal', 'Status Pembayaran']

def generate_inisials(n, seed=42):
    random.seed(seed)
    used = set()
    hasil = []
    while len(hasil) < n:
        inisial = ''.join(random.choices(string.ascii_uppercase, k=2))
        if inisial not in used:
            used.add(inisial)
            hasil.append(inisial)
    return hasil

warga_asli = sorted(df['Nama'].unique())
inisials = generate_inisials(len(warga_asli))
anon_map = {asli: inisial for asli, inisial in zip(warga_asli, inisials)}
df['Nama'] = df['Nama'].map(anon_map)

df['Tanggal Bayar'] = df['Tanggal Bayar'].replace('-', None)
df['Tanggal Bayar'] = pd.to_datetime(df['Tanggal Bayar'], format='%d-%m-%Y', errors='coerce')
df['Tanggal'] = df['Tanggal Bayar'].dt.day

df_tampil = df.copy()
df_tampil['Tanggal Bayar'] = df_tampil['Tanggal Bayar'].dt.strftime('%d-%m-%Y').fillna('-')

print(f'Total warga: {df["Nama"].nunique()}')
print(f'Total baris: {len(df)}')
print('\nTabel 1 - Contoh Data Mentah (15 Baris Pertama):')
df_tampil[['Nama','Tanggal Bayar','Nominal','Status Pembayaran']].head(15)

In [ ]:
print('=== CEK MISSING VALUES ===')
print(df[['Nama','Nominal','Status Pembayaran']].isnull().sum())
print(f'\n=== CEK DUPLIKAT ===')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')
print(f'\n=== VALIDASI NILAI STATUS ===')
vals = df['Status Pembayaran'].unique()
print(f'Nilai unik: {vals} → OK')
print(f'\n=== DISTRIBUSI STATUS ===')
print(f'Sudah bayar (1): {(df["Status Pembayaran"]==1).sum()} transaksi')
print(f'Belum bayar (0): {(df["Status Pembayaran"]==0).sum()} transaksi')
print('\n✓ Data bersih, siap untuk transformasi RFM.')

In [ ]:
rfm_list = []
for nama in df['Nama'].unique():
    w = df[df['Nama'] == nama].copy()
    frequency = w['Status Pembayaran'].sum()
    monetary  = w[w['Status Pembayaran'] == 1]['Nominal'].sum()
    tgl_bayar = w[w['Status Pembayaran'] == 1]['Tanggal Bayar'].dropna()
    if len(tgl_bayar) > 0:
        last = tgl_bayar.max()
        recency = (PERIODE_AKHIR.year - last.year)*12 + (PERIODE_AKHIR.month - last.month)
    else:
        recency = 12
    tgl_vals = w[w['Status Pembayaran'] == 1]['Tanggal'].dropna()
    avg_tgl  = round(tgl_vals.mean()) if len(tgl_vals) > 0 else None
    rfm_list.append({'Nama': nama, 'Recency': recency,
                     'Frequency': frequency, 'Monetary': monetary,
                     'Avg_Tgl_Bayar': avg_tgl})

df_rfm = pd.DataFrame(rfm_list)

print('Tabel 2 - Contoh Hasil Transformasi RFM (10 Warga Pertama):')
df_rfm[['Nama','Recency','Frequency','Monetary']].head(10)

In [ ]:
rfm_features = ['Recency', 'Frequency', 'Monetary']
scaler = MinMaxScaler()
X = scaler.fit_transform(df_rfm[rfm_features])

df_rfm_norm = df_rfm.copy()
df_rfm_norm[rfm_features] = X

print('Tabel 3 - Data Setelah Normalisasi Min-Max (10 Warga Pertama):')
df_rfm_norm[['Nama','Recency','Frequency','Monetary']].head(10).round(4)

In [ ]:
desc = df_rfm[rfm_features].describe().round(6)
desc.index = ['Count','Mean','Std Dev','Minimum',
              'Q1 (25%)','Median (50%)','Q3 (75%)','Maximum']
desc.columns = ['Recency (Bulan)', 'Frequency (Bulan)', 'Monetary (Rupiah)']

print(f'Tabel 4 - Statistik Deskriptif Variabel RFM (n={len(df_rfm)} warga):')
desc

In [ ]:
hasil_elbow = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, init='k-means++', max_iter=100,
                n_init=10, random_state=42)
    km.fit(X)
    sil = silhouette_score(X, km.labels_)
    hasil_elbow.append({'K': k, 'WCSS': round(km.inertia_, 4),
                        'Silhouette Score': round(sil, 4)})

df_elbow = pd.DataFrame(hasil_elbow)
print('Tabel 5 - Hasil Perhitungan WCSS dan Silhouette Score:')
display(df_elbow)

# Gambar 3
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(df_elbow['K'], df_elbow['WCSS'], 'o-', color='#2196F3',
         linewidth=2.5, markersize=8, label='WCSS')
ax1.set_xlabel('Jumlah Cluster (K)', fontsize=12)
ax1.set_ylabel('WCSS', color='#2196F3', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#2196F3')
ax1.set_xticks(range(2, 11))
elbow_wcss = df_elbow[df_elbow['K']==3]['WCSS'].values[0]
ax1.annotate('Elbow Point\n(K=3)', xy=(3, elbow_wcss), xytext=(4.2, elbow_wcss+1.5),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=10, color='red', fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(df_elbow['K'], df_elbow['Silhouette Score'], 's--', color='#FF5722',
         linewidth=2, markersize=7, label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='#FF5722', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#FF5722')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper right')
plt.title('Elbow Method untuk Penentuan K Optimal',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=100,
                n_init=10, random_state=42)
kmeans.fit(X)
df_rfm['Cluster'] = kmeans.labels_

cluster_freq = df_rfm.groupby('Cluster')['Frequency'].mean().sort_values(ascending=False)
label_map = {
    cluster_freq.index[0]: 'Warga Disiplin',
    cluster_freq.index[1]: 'Warga Tidak Konsisten',
    cluster_freq.index[2]: 'Warga Berisiko Tinggi'
}
df_rfm['Label'] = df_rfm['Cluster'].map(label_map)
urutan = ['Warga Disiplin', 'Warga Tidak Konsisten', 'Warga Berisiko Tinggi']

print(f'✓ K-Means selesai. Iterasi: {kmeans.n_iter_}')
print(f'  WCSS Final: {kmeans.inertia_:.4f}')

# Tabel 6 - Centroid
centroids_asli = scaler.inverse_transform(kmeans.cluster_centers_)
rows = []
for i, row in enumerate(centroids_asli):
    n = len(df_rfm[df_rfm['Cluster']==i])
    rows.append({'Cluster': i, 'Label': label_map[i],
                 'Recency': round(row[0],3), 'Frequency': round(row[1],3),
                 'Monetary (Rp)': round(row[2],0),
                 'Jumlah Warga': n, 'Persentase': f"{n/214*100:.2f}%"})
df_centroid = pd.DataFrame(rows)
df_centroid['urutan'] = df_centroid['Label'].map({v:i for i,v in enumerate(urutan)})
df_centroid = df_centroid.sort_values('urutan').drop('urutan',axis=1).reset_index(drop=True)
print('\nTabel 6 - Centroid Akhir Tiap Cluster (Nilai Asli):')
display(df_centroid)

# Tabel 7 - Distribusi
dist = [{'Label': lbl, 'Jumlah Warga': len(df_rfm[df_rfm['Label']==lbl]),
         'Persentase': f"{len(df_rfm[df_rfm['Label']==lbl])/214*100:.2f}%"}
        for lbl in urutan]
print('\nTabel 7 - Distribusi Warga per Cluster:')
pd.DataFrame(dist)

In [ ]:
print('Tabel 8 - Contoh Hasil Clustering (15 Warga Pertama):')
df_rfm[['Nama','Recency','Frequency','Monetary','Cluster','Label']].head(15)

In [ ]:
df_rfm['Jarak_ke_Centroid'] = 0.0
for c in range(3):
    idx = df_rfm['Cluster'] == c
    data_c = X[idx]
    jarak = np.sqrt(((data_c - kmeans.cluster_centers_[c])**2).sum(axis=1))
    df_rfm.loc[idx, 'Jarak_ke_Centroid'] = jarak.round(4)

print('Rata-rata Jarak Euclidean ke Centroid per Cluster:')
for lbl in urutan:
    avg = df_rfm[df_rfm['Label']==lbl]['Jarak_ke_Centroid'].mean()
    print(f'  {lbl}: {avg:.4f}')

In [ ]:
color_map = {'Warga Disiplin':'#4CAF50',
             'Warga Tidak Konsisten':'#FF9800',
             'Warga Berisiko Tinggi':'#F44336'}

sizes = [len(df_rfm[df_rfm['Label']==lbl]) for lbl in urutan]
labels_pie = [f"{lbl}\n({n} warga, {n/214*100:.2f}%)"
              for lbl, n in zip(urutan, sizes)]
colors_pie = [color_map[lbl] for lbl in urutan]

fig, ax = plt.subplots(figsize=(9, 6))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels_pie, colors=colors_pie,
    autopct='%1.2f%%', startangle=140,
    textprops={'fontsize': 10},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')
ax.set_title('Distribusi Warga Berdasarkan Cluster\nKomplek KEPU Periode 2025',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for lbl in urutan:
    s = df_rfm[df_rfm['Label'] == lbl]
    ax.scatter(s['Frequency'], s['Monetary'], c=color_map[lbl],
               label=lbl, alpha=0.7, s=60, edgecolors='white', linewidth=0.5)
for i, row in enumerate(centroids_asli):
    ax.scatter(row[1], row[2], c='black', marker='X', s=200, zorder=5)
    ax.annotate(f"C{i}", (row[1], row[2]), textcoords='offset points',
                xytext=(8, 5), fontsize=9, fontweight='bold')
ax.set_xlabel('Frequency (Jumlah Bulan Bayar)', fontsize=12)
ax.set_ylabel('Monetary (Total Pembayaran Rp)', fontsize=12)
ax.set_title('Visualisasi 2D: Frequency vs Monetary',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection='3d')
for lbl in urutan:
    s = df_rfm[df_rfm['Label'] == lbl]
    ax.scatter(s['Recency'], s['Frequency'], s['Monetary'],
               c=color_map[lbl], label=lbl, alpha=0.7, s=50)
for row in centroids_asli:
    ax.scatter(row[0], row[1], row[2], c='black', marker='X', s=200, zorder=5)
ax.set_xlabel('Recency (Bulan)', fontsize=10)
ax.set_ylabel('Frequency (Bulan)', fontsize=10)
ax.set_zlabel('Monetary (Rp)', fontsize=10)
ax.set_title('Visualisasi 3D: Recency, Frequency, Monetary',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Hitung jarak Euclidean setiap warga ke centroid cluster-nya
from sklearn.metrics import pairwise_distances

jarak_list = []

for nama in df_rfm['Nama'].unique():
    # Ambil data warga ini (sudah dinormalisasi)
    warga_norm = df_rfm_norm[df_rfm_norm['Nama'] == nama][rfm_features].values
    
    # Ambil cluster warga ini
    cluster_warga = df_rfm[df_rfm['Nama'] == nama]['Cluster'].values[0]
    
    # Ambil centroid dari cluster tersebut
    centroid = kmeans.cluster_centers_[cluster_warga]
    
    # Hitung jarak Euclidean
    jarak = pairwise_distances(warga_norm, [centroid])[0][0]
    jarak_list.append(jarak)

# Tambahkan kolom jarak ke DataFrame
df_rfm['Jarak_ke_Centroid'] = jarak_list

print("✓ Kolom Jarak_ke_Centroid berhasil dihitung")
fig, ax = plt.subplots(figsize=(9, 5))
data_box, label_box, colors_box = [], [], []
for lbl in urutan:
    c_id = [k for k, v in label_map.items() if v == lbl][0]
    vals = df_rfm[df_rfm['Cluster'] == c_id]['Jarak_ke_Centroid'].values
    data_box.append(vals)
    label_box.append(lbl.replace(' ', '\n'))
    colors_box.append(color_map[lbl])
bp = ax.boxplot(data_box, labels=label_box, patch_artist=True)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for i, vals in enumerate(data_box, 1):
    avg = round(np.mean(vals), 4)
    ax.text(i, avg + 0.003, f'Avg: {avg}', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Jarak Euclidean ke Centroid', fontsize=12)
ax.set_title('Visualisasi Jarak Data ke Centroid per Cluster',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
for i, lbl in enumerate(urutan):
    subset = df_rfm[df_rfm['Label'] == lbl][rfm_features]
    desc = subset.describe().round(3)
    desc.index = ['Count','Mean','Std Dev','Min','Q1 (25%)','Median','Q3 (75%)','Max']
    print(f'Tabel {9+i} - Statistik {lbl} (n={len(subset)} warga):')
    display(desc)
    print()

In [ ]:
rows_perb = []
row = {'Variabel': '% Populasi'}
for lbl in urutan:
    n = len(df_rfm[df_rfm['Label']==lbl])
    row[lbl] = f"{n} warga ({n/214*100:.2f}%)"
rows_perb.append(row)
for v in rfm_features:
    row = {'Variabel': v}
    for lbl in urutan:
        s = df_rfm[df_rfm['Label']==lbl][v]
        row[lbl] = f"Mean: {s.mean():.2f} | Range: {s.min():.0f}-{s.max():.0f}"
    rows_perb.append(row)

print('Tabel 12 - Perbandingan Komprehensif Antar Cluster:')
pd.DataFrame(rows_perb)

In [ ]:
# ==========================================
# ANALISIS TANGGAL PEMBAYARAN PER CLUSTER
# ==========================================

# Definisikan urutan cluster yang benar (Cluster 2 = Disiplin, Cluster 1 = Tidak Konsisten, Cluster 0 = Berisiko)
urutan_benar = ['Warga Disiplin', 'Warga Tidak Konsisten', 'Warga Berisiko Tinggi']

# Urutan warna yang sesuai dengan cluster (Hijau untuk Disiplin, Orange untuk Tidak Konsisten, Merah untuk Berisiko)
warna_benar = ['#4CAF50', '#FF9800', '#F44336']

rows_tgl = []

for lbl in urutan_benar:
    nama_cluster = df_rfm[df_rfm['Label'] == lbl]['Nama'].values
    tgl_vals = df[df['Nama'].isin(nama_cluster) & (df['Status Pembayaran'] == 1)]['Tanggal'].dropna()
    
    if len(tgl_vals) > 0:
        rows_tgl.append({
            'Cluster': lbl,
            'Rata-rata Tgl Bayar': round(tgl_vals.mean(), 1),
            'Tgl Paling Sering': int(tgl_vals.mode()[0]) if len(tgl_vals.mode()) > 0 else None,
            'Tgl Minimum': int(tgl_vals.min()),
            'Tgl Maksimum': int(tgl_vals.max()),
            'Periode Dominan': f"Tgl {int(tgl_vals.quantile(0.25))}-{int(tgl_vals.quantile(0.75))}"
        })

print('Tabel 13 - Analisis Tanggal Pembayaran per Cluster:')
display(pd.DataFrame(rows_tgl))

# Gambar Histogram 3 Cluster (dengan urutan yang benar)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, (lbl, warna) in enumerate(zip(urutan_benar, warna_benar)):
    nama_cluster = df_rfm[df_rfm['Label'] == lbl]['Nama'].values
    tgl_vals = df[df['Nama'].isin(nama_cluster) & (df['Status Pembayaran'] == 1)]['Tanggal'].dropna()
    
    if len(tgl_vals) > 0:
        axes[i].hist(tgl_vals, bins=range(1, 33), color=warna,
                     alpha=0.8, edgecolor='white')
        axes[i].axvline(tgl_vals.mean(), color='black', linestyle='--', linewidth=1.5,
                        label=f'Mean: {tgl_vals.mean():.1f}')
        axes[i].set_title(f'{lbl}\n(n={len(nama_cluster)} warga)', fontsize=10, fontweight='bold')
        axes[i].set_xlabel('Tanggal Pembayaran', fontsize=9)
        axes[i].set_ylabel('Frekuensi', fontsize=9)
        axes[i].set_xticks([1, 5, 10, 15, 20, 25, 30])
        axes[i].legend(fontsize=8)
        axes[i].grid(True, alpha=0.3, axis='y')

plt.suptitle('Distribusi Tanggal Pembayaran per Cluster',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
sil = silhouette_score(X, kmeans.labels_)
db  = davies_bouldin_score(X, kmeans.labels_)
ch  = calinski_harabasz_score(X, kmeans.labels_)

df_eval = pd.DataFrame([
    {'Metrik': 'Silhouette Coefficient', 'Nilai': round(sil, 4),
     'Threshold': '> 0.5 = Baik', 'Hasil': 'Baik ✓' if sil > 0.5 else 'Perlu Evaluasi'},
    {'Metrik': 'Davies-Bouldin Index', 'Nilai': round(db, 4),
     'Threshold': '< 1.0 = Baik', 'Hasil': 'Baik ✓' if db < 1 else 'Perlu Evaluasi'},
    {'Metrik': 'Calinski-Harabasz Index', 'Nilai': round(ch, 2),
     'Threshold': 'Semakin tinggi semakin baik', 'Hasil': 'Baik ✓'},
    {'Metrik': 'WCSS (Inertia)', 'Nilai': round(kmeans.inertia_, 4),
     'Threshold': 'Semakin kecil semakin baik', 'Hasil': 'Optimal ✓'},
])

print('Tabel 14 - Evaluasi Kuantitatif Hasil Clustering:')
df_eval

In [ ]:
from scipy import stats

# Pisahkan data per cluster
c0 = df_rfm[df_rfm['Cluster'] == 0][['Recency', 'Frequency', 'Monetary']]
c1 = df_rfm[df_rfm['Cluster'] == 1][['Recency', 'Frequency', 'Monetary']]
c2 = df_rfm[df_rfm['Cluster'] == 2][['Recency', 'Frequency', 'Monetary']]

for var in ['Recency', 'Frequency', 'Monetary']:
    # Uji Normalitas
    for name, grp in zip(['C0','C1','C2'], [c0, c1, c2]):
        w, p = stats.shapiro(grp[var])
        print(f"Shapiro {var} {name}: W={w:.4f}, p={p:.4f}")

    # Kruskal-Wallis
    h, p = stats.kruskal(c0[var], c1[var], c2[var])
    print(f"Kruskal {var}: H={h:.4f}, p={p:.6f}")

    # Effect Size Eta Squared
    all_data = df_rfm[var]
    grand_mean = all_data.mean()
    ss_between = sum(len(g[var]) * (g[var].mean() - grand_mean)**2 
                     for g in [c0, c1, c2])
    ss_total = ((all_data - grand_mean)**2).sum()
    print(f"Eta Squared {var}: {ss_between/ss_total:.4f}")

    # Post-hoc Mann-Whitney
    for label, ga, gb in [("C2 vs C1", c2[var], c1[var]),
                           ("C2 vs C0", c2[var], c0[var]),
                           ("C1 vs C0", c1[var], c0[var])]:
        u, p = stats.mannwhitneyu(ga, gb, alternative='two-sided')
        print(f"Mann-Whitney {var} {label}: U={u:.1f}, p={p:.6f}")
    print()